In [3]:
#Tools
from langchain_core.tools import tool
@tool
def add_numbers(a: float, b: float) -> float:
    """add two number"""
    return a + b

@tool
def get_weather(city: str) -> dict:
    """get weather report of a city"""
    return {
        "city": city,
        "temperature_c": 28,
        "condition": "Cloudy"
    }


tools =[add_numbers,get_weather]




In [4]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-5-nano",temperature=0)

llm_with_tools=llm.bind_tools(tools)



In [5]:
response = llm_with_tools.invoke(
    "What is 15 + 27? use the calcualtor build"
)

print(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 161, 'total_tokens': 316, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EZyaWqu2IciEQBYGgd409AjUUmG8C', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a14a03-32b6-7f02-8c06-0cd8e694584a-0' tool_calls=[{'name': 'add_numbers', 'args': {'a': 15, 'b': 27}, 'id': 'call_bZwR7LnLZP0YgqZSb2CA29DS', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 161, 'output_tokens': 155, 'total_tokens': 316, 'input_token_details': {'audio': 0, 'cache_read': 0

In [6]:
from langchain_core.messages import ToolMessage
tool_registry = {
    tool.name: tool for tool in tools
}


messages = [
    ("user", "What is 15 + 27? Use the calculator tool.")
]

response = llm_with_tools.invoke(messages)
messages.append(response)

for tool_call in response.tool_calls:
    result = tool_registry[tool_call["name"]].invoke(
        tool_call["args"]
    )
    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"],
        )
    )

final_response = llm_with_tools.invoke(messages)
print(final_response.content)

The result is 42.0. If you’d like, I can show the calculation steps or use a different operation.


In [ ]:


from langchain_core.messages import HumanMessage, ToolMessage

messages = [
    HumanMessage(
        content="What is 15 + 27 ? what is the weather today in Buaner"
    )
]

MAX_STEPS = 1

for step in range(MAX_STEPS):
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        print("Final answer:", response.content)
        break
    print(f"tools called total {len(response.tool_calls)}")
    
    for tool_call in response.tool_calls:
        name = tool_call["name"]
        args = tool_call["args"]
        print(f"tool name : {name}")
        if name not in tool_registry:
            raise ValueError(f"Unknown tool: {name}")

        result = tool_registry[name].invoke(args)

        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"],
            )
        )
else:
    print(llm_with_tools.invoke(messages).content)




tools called total 1
tool name : get_weather
- 15 + 27 = 42
- Weather in Buaner today: 28°C, Cloudy


In [ ]:

from langchain_openai import ChatOpenAI
from langchain_core.messages import ToolMessage
from computer_tools import list_files, read_text_file

tools = [list_files, read_text_file]
tool_registry = {tool.name: tool for tool in tools}

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

llm_with_tools = llm.bind_tools(tools)

messages = [
    ("user", "Find the files in my workspace and read notes.txt.")
]

while True:
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        print("\nFinal answer:")
        print(response.content)
        break

    
    for call in response.tool_calls:
        print("Calling:", call["name"], call["args"])

        tool = tool_registry[call["name"]]

        try:
            result = tool.invoke(call["args"])
            print("SUCCESS:", repr(result))
        except Exception as exc:
            result = f"Tool error: {type(exc).__name__}: {exc}"
            print("FAILED:", result)

        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"],
            )
        )




Calling: list_files {}
SUCCESS: ['notes.txt']
Calling: read_text_file {'relative_path': 'notes.txt'}
SUCCESS: 'I am learning agentic AI and building a local computer assistant.'

Final answer:
I found the file `notes.txt`, and it contains the following text:

"I am learning agentic AI and building a local computer assistant."


Expected workspace: C:\Users\mukes\Documents\multi-agent-system\src\agent\agent_workspace
Directory exists: True
Files: ['notes.txt']
